# 1. Product extractor
In this task you will based on product descriptions, be able to extract out relevant information into structured format using pydantic model together with PydanticAI agent.

a) Read the products.json and store all of its descriptions into a list

In [27]:
import json

with open("data/products.json", "r") as file:
    products = json.load(file)

product_descriptions = [product["description"] for product in products]

product_descriptions


['The Nike Running Shoes X90 are available for $120.00 in the Footwear section.',
 'Grab the Sony WH-1000XM5 Headphones for just $349.99 — found in the Electronics department.',
 'The Dyson V12 Vacuum Cleaner is out of stock but available for pre-order at $599.99 in the Home & Garden section.',
 'Priced at $189.95, the Adidas Ultraboost 22 Sneakers are available in our Footwear collection.',
 'You can find the Apple iPad Pro 11-inch for $799.00 in the Electronics aisle.',
 "The Levi's 501 Original Jeans are currently available for $69.50 under our Clothing department.",
 'The Bosch Cordless Drill GSR18 is out of stock but available for pre-order at $149.00 in the Automotive section.',
 'Grab the Canon EOS R50 Camera for just $679.99 — available in the Electronics section.',
 'Priced at $299.95, the Philips Air Purifier 3000i is available in our Home & Garden collection.',
 'The Samsung Galaxy Watch 6 is out of stock but available for pre-order at $299.00 in the Electronics department.'

b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

In [32]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

class Product(BaseModel):
    name: str = Field(description="The name of the product")
    price: float = Field(description="The price of the product")
    category: str = Field(description="The category of the product")
    in_stock: bool = Field(description="Whether the product is in stock")
    description: str = Field(description="The description of the product")

product_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt="You are a product extractor. You will be given a product description and you will need to extract the structured output from the description.",
    retries=1,
)
products = []

for desc in product_descriptions:
    results = await product_agent.run(desc, output_type=Product)
    products.append(results.output)

products


[Product(name='Nike Running Shoes X90', price=120.0, category='Footwear', in_stock=True, description='The Nike Running Shoes X90 are available for $120.00 in the Footwear section.'),
 Product(name='Sony WH-1000XM5 Headphones', price=349.99, category='Electronics', in_stock=True, description='Grab the Sony WH-1000XM5 Headphones for just $349.99 — found in the Electronics department.'),
 Product(name='Dyson V12 Vacuum Cleaner', price=599.99, category='Home & Garden', in_stock=False, description='The Dyson V12 Vacuum Cleaner is out of stock but available for pre-order at $599.99 in the Home & Garden section.'),
 Product(name='Adidas Ultraboost 22 Sneakers', price=189.95, category='Footwear', in_stock=True, description='Priced at $189.95, the Adidas Ultraboost 22 Sneakers are available in our Footwear collection.'),
 Product(name='Apple iPad Pro 11-inch', price=799.0, category='Electronics', in_stock=True, description='You can find the Apple iPad Pro 11-inch for $799.00 in the Electronics 

c) Now try to validate programmatically if the output fields are correct.

In [34]:
import re

for p in products:
    expected_price = float(re.search(r"\$(\d+(?:\.\d{1,2})?)", p.description).group(1))
    expected_stock = "out of stock" not in p.description.lower()

    ok = (
        p.name in p.description
        and p.category in p.description
        and p.price == expected_price
        and p.in_stock == expected_stock
    )

    print(f"{p.name}: {'OK' if ok else 'FAIL'}")

Nike Running Shoes X90: OK
Sony WH-1000XM5 Headphones: OK
Dyson V12 Vacuum Cleaner: OK
Adidas Ultraboost 22 Sneakers: OK
Apple iPad Pro 11-inch: OK
Levi's 501 Original Jeans: OK
Bosch Cordless Drill GSR18: OK
Canon EOS R50 Camera: OK
Philips Air Purifier 3000i: OK
Samsung Galaxy Watch 6: OK
